All necessary inclusions. First, run this cell.

In [ ]:
#include <vector>
#include <iostream>
#include <sstream>
#pragma  cling add_include_path("../../lib/")
#include <walimpl.hpp>
#include <syncmem.hpp>

The ***mapper*** class allows you to link an arbitrary *algorithm* for element-by-element processing of a data set (the *algorithm* inherits the ***mapper*** class) and an arbitrary implementation of the *mechanism* for performing element-by-element processing (the *mechanism* for performing element-by-element processing inherits the ***mapper::engine*** class).

In [ ]:
class mapper{
public:
    class engine{
    friend class mapper;
    protected:
        virtual void init(unsigned size) = 0;
        virtual void map() = 0;
    protected:
        void on_init(unsigned size){_mapper->on_init(size);}
        void on_map(unsigned id){_mapper->on_map(id);}
    	void on_save(unsigned id, std::ostream&out, bool mapped){
            _mapper->on_save(id,out,mapped);
        }
    	void on_load(unsigned id, std::istream&in, bool mapped){
            _mapper->on_load(id,in,mapped);
        }
    protected:
        mapper* _mapper;
    };
public:
    mapper(mapper::engine& eng):_engine(eng){eng._mapper=this;}
    void map(){_engine.map();}
    void init(unsigned size){_engine.init(size);}
protected:
    virtual void on_init(unsigned size) = 0;
    virtual void on_map(unsigned id) = 0;
	virtual void on_save(unsigned id, std::ostream&, bool mapped) = 0;
	virtual void on_load(unsigned id, std::istream&, bool mapped) = 0;
private:
    mapper::engine& _engine;
};

The ***basic_engine*** class illustrates a basic sequential implementation of element-by-element processing of an arbitrary data array. In addition to calling the *on_map(id)* method for processing an array element, a test for uploading/downloading of raw and processed array elements is performed (*void io_test(unsigned id,bool mapped){...}*).

In [ ]:
class basic_engine: public mapper::engine{
    void init(unsigned size)override
            {_size=size;on_init(size);}
    void map()override{
        for(int id=0;id<_size;id++)
            {io_test(id,false); on_map(id); io_test(id,true);}
    }
    void io_test(unsigned id,bool mapped){
        std::stringstream sstr;
        on_save(id,sstr,mapped);on_load(id,sstr,mapped);
    }
    unsigned _size;
};

The ***sync_state_engine class*** implements a *mechanism* for element-by-element processing using a synchronized global state. The state of all ***templet::state*** class objects is synchronized via the event log.

In [ ]:
class sync_state_engine: public mapper::engine{
public:
    sync_state_engine():_tbag(_wal,*this){}
private:
    void init(unsigned size)override{
        _tbag.resize(size);
        for(int id=0; id<size; id++)_tbag.add(id);
    }
    void map()override{
        while(!_tbag.ready_to_get())/*wait*/;
        unsigned id;
        while(_tbag.get(id)){on_map(id);_tbag.put(id);}
    }
    class taskbag:public templet::state{
    friend class sync_state_engine;
        taskbag(templet::write_ahead_log&l,sync_state_engine&eng):
            state(l),_engine(eng) {init();}
        void resize(unsigned size){
            update(_resize, [&](std::ostream&out) {
                out << size;
    		},
    		[this](std::istream&in) {
                unsigned size; in >> size;
                _size = size; _engine.on_init(size);
                unprocessed.clear();
    		});   
        }
        void add(unsigned id){
            update(_add, [&](std::ostream&out) {
                out << id << " "; _engine.on_save(id,out,false);
    		},
    		[this](std::istream&in) {
                unsigned id; in >> id;
                _engine.on_load(id,in,false);
                unprocessed.insert(id);
    		});     
        }
        bool ready_to_get(){
            update();
            return unprocessed.size()==_size;
        }
        bool get(unsigned& id){
            update(); 
            if(unprocessed.empty())return false;
            id = get_rand_unprocessed();  
            return true;
        }
        void put(unsigned id){
            update(_put, [&](std::ostream&out) {
                out << id << " "; _engine.on_save(id,out,true);
    		},
    		[this](std::istream&in) {
                unsigned id; in >> id; _engine.on_load(id,in,true);
                unprocessed.erase(id);
    		});  
        }
        enum {_resize,_add,_put};
    	void on_init() override {resize(0); add(0); put(0);}
        sync_state_engine& _engine;
        unsigned _size;
        std::set<unsigned> unprocessed;
        unsigned get_rand_unprocessed(){
            int selected = rand() % unprocessed.size();
    		auto it = unprocessed.begin(); 
            for (int i = 0; i != selected; i++, it++) {}
            return *it;
        }
    } _tbag;
    templet::write_ahead_log _wal;
};

The ***everest_engine*** class implements a *mechanism* for element-by-element processing using the Everest platform for distributed computing (https://everest.distcomp.org).

In [ ]:
class everest_engine: public mapper::engine{
    void init(unsigned size)override
            {_size=size;on_init(size);}
    void map()override{
        for(int id=0;id<_size;id++)
            {io_test(id,false); on_map(id); io_test(id,true);}
    }
    void io_test(unsigned id,bool mapped){
        std::stringstream sstr;
        on_save(id,sstr,mapped);on_load(id,sstr,mapped);
    }
    unsigned _size;
};

The ***throughput_test_mapper*** test squares each element of the original *N*-array. The result is written to an *NxN* array. Since the operation takes little time, the test allows us to estimate the overhead of running an operation to process an element of the data array. Additionally, the test verifies the correctness of the *algorithm's* implementation and execution *mechanism*.

In [ ]:
class throughput_test_mapper: public mapper{
public:
    throughput_test_mapper(mapper::engine& eng): mapper(eng){}
private:
    void on_init(unsigned size) override{
        N.resize(size); NxN.resize(size);
    }
    void on_map(unsigned id) override{
        NxN[id] = N[id] * N[id];
    }
    void on_save(unsigned id, std::ostream& out, bool mapped) override{
        if(mapped) out << NxN[id]; else out << N[id];  
    }
	void on_load(unsigned id, std::istream& in, bool mapped) override{
        if(mapped) in >> NxN[id]; else in >> N[id];
    }
public:
    std::vector<int> N;
    std::vector<int> NxN;
};

The method of testing multiple implementations (***basic_engine***, ***sync_state_engine***, ***everest_engine***) of the algorithm for element-by-element squaring of numeric array elements. The *if(master){..}* sections for input data and output of processing results are executed in the main process (*master==true*).

In [ ]:
{
    const int SIZE = 10;
    bool master = true;
    
    //basic_engine eng;
    sync_state_engine eng;
    //everest_engine eng;
    throughput_test_mapper a_mapper(eng);
    
    if(master){
        a_mapper.N.resize(SIZE);
        for(int i=0;i<SIZE;i++) a_mapper.N[i] = i;
        a_mapper.init(SIZE);
    }
    
    a_mapper.map();
   
    if(master){
        for(int i=0;i<SIZE;i++)
            std::cout << a_mapper.N[i] <<"^2 = " << a_mapper.NxN[i] << std::endl;
    }
}